# AWS Bedrock Flows: Visual Workflow Orchestration

**Reference:** Slide 38 (*Hands-On RAG with AWS*)

**AWS Bedrock Flows** allows you to build multi-step generative AI workflows using a visual Directed Acyclic Graph (DAG) interface and declarative APIs (`boto3.client('bedrock-agent')`).

### Supported Node Types Illustrated in this Demo:
1. **Input Node (`FlowInput`)**: Receives the user question or triggering event.
2. **Knowledge Base Node (`RetrieveKnowledgeBase`)**: Executes vector similarity retrieval against Bedrock Knowledge Bases.
3. **Condition Node (`CheckRetrievalScore`)**: Branching logic evaluating whether retrieved context meets relevance thresholds.
4. **Prompt Node (`SynthesizeAnswerPrompt`)**: Invokes foundation models with prompt templates and dynamic variable injection.
5. **Output Node (`FlowOutput`)**: Formats and returns the final response.

In [ ]:
from bedrock_flows_demo import BedrockFlowManager

# Initialize Bedrock Flow Manager (auto-detects credentials or falls back to mock mode)
manager = BedrockFlowManager(region_name="us-east-1")
print(f"Flow Manager initialized. Mock Mode: {manager.mock_mode}")

## 1. Construct the Multi-Node Flow Definition
We declare nodes and explicit directional data connections.

In [ ]:
definition = manager.build_rag_flow_definition(
    kb_id="SAMPLE_KB_12345",
    model_id="anthropic.claude-3-haiku-20240307-v1:0"
)

print(f"Defined Nodes ({len(definition['nodes'])}):")
for n in definition['nodes']:
    print(f" - [{n['type']:<15}] {n['name']}")

print(f"\nDefined Connections ({len(definition['connections'])}):")
for c in definition['connections']:
    print(f" - {c['source']} --> {c['target']}")

## 2. Validate Flow Topology & Schema
Before deployment, validate that input/output connections are complete and no orphaned nodes exist.

In [ ]:
validation = manager.validate_flow(definition)
print(f"Validation Passed: {validation['valid']}")
print(f"Node Count: {validation['node_count']}, Connection Count: {validation['connection_count']}")

## 3. Create & Prepare Flow (`create_flow`, `prepare_flow`)
Deploy the flow to Bedrock Agent service and compile it for execution.

In [ ]:
import time
flow_name = f"demo-rag-flow-{int(time.time())}"
flow = manager.create_flow(name=flow_name, definition=definition)
flow_id = flow.get("id")
print(f"Flow Created. ID: {flow_id}, Status: {flow.get('status')}")

prep = manager.prepare_flow(flow_id)
print(f"Flow Prepared for Execution. Status: {prep.get('status')}")

## 4. Infrastructure as Code (CDK / CloudFormation)
Inspect the CloudFormation representation generated from CDK.

In [ ]:
from bedrock_flow_cdk import generate_cloudformation_template
import json

cfn = generate_cloudformation_template()
print(json.dumps(cfn, indent=2))

## 5. Teardown

In [ ]:
manager.delete_flow(flow_id)
print(f"Flow {flow_id} cleaned up successfully.")